# 02 · Analisis alert

Pertanyaan yang dijawab: **kapan, di mana, dan seberapa cepat** pelanggaran ditangani. Fungsi di `slopdash.metrics` dan `slopdash.charts` dipakai ulang persis oleh dashboard Streamlit.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import plotly.io as pio

from dashboard import charts, data, metrics

pio.renderers.default = "notebook_connected"
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
devices = data.load_devices()
alerts = data.load_alerts()
metrics.kpis(alerts, devices)

## Tren harian

In [ ]:
charts.fig_alerts_per_day(metrics.alerts_per_day(alerts))

## Kapan pelanggaran paling sering terjadi? (hari × jam)

In [ ]:
grid = metrics.alert_heatmap(alerts)
charts.fig_heatmap(grid)

In [ ]:
jam_puncak = grid.sum().sort_values(ascending=False).head(3)
print('3 jam tersibuk:', ', '.join(f'{j:02d}:00 ({v} alert)' for j, v in jam_puncak.items()))

## Unit dan lokasi

In [ ]:
metrics.alerts_by_unit(alerts)

In [ ]:
charts.fig_hotspots(metrics.alert_hotspots(alerts))

## Kecepatan penanganan

`time_to_resolve_min` = waktu dari alert pertama sampai ditandai selesai oleh petugas.

In [ ]:
alerts.groupby('device_name')['time_to_resolve_min'].describe()[['count', '50%', 'mean', 'max']]

In [ ]:
charts.fig_duration_hist(alerts)

## Alert yang masih terbuka lebih dari 1 hari

In [ ]:
kini = pd.Timestamp.now(tz='Asia/Jakarta').tz_localize(None)
terbuka = alerts[~alerts['is_resolved']].assign(umur_jam=lambda d: (kini - d['created_at']).dt.total_seconds() / 3600)
terbuka[terbuka['umur_jam'] > 24][['created_at', 'unit_label', 'alert_type', 'umur_jam']]